# Deep Research — plan, research, synthesize with citations (OpenAI-style)

[OpenAI Deep Research](https://openai.com/index/introducing-deep-research/)
decomposes a question into sub-questions, runs independent research threads on
each, and synthesizes a long-form cited report. The pipeline makes the output
trustworthy: each stage's output is a typed schema, and the synthesizer sees
only source-attributed findings — it structurally cannot make an uncited claim.

```
question ──▶ PLANNER ──▶ sub-questions ──▶ RESEARCHER (×N) ──▶ findings ──▶ SYNTHESIZER ──▶ cited report
             output_schema=                tools: search +       output_schema=
             ResearchPlan                  read_article          ResearchNotes
```

**SDK primitives this notebook showcases:**
- `output_schema` + Pydantic models — typed stage boundaries; `reply.metadata["structured"]` carries the validated payload
- `@tool` for lightweight search/read wrappers over Wikipedia's public API
- `TokenBudgetMiddleware`, `CostBudgetMiddleware`, `RuntimeLimitMiddleware` — cost control per research agent
- `SequentialPipeline` — wiring planner → researcher(s) → synthesizer as a composable pipeline

> **Before running:** copy `.env.example` to `.env` at the repo root and add your provider key.

## 1. Environment & constants

| Constant | What it controls |
|---|---|
| `PROVIDER` / `MODEL` | LLM for all three stages. Swap to `anthropic` / `claude-sonnet-4-6` for a different model. |
| `MAX_TOOL_ROUNDS` | How many search→read cycles each researcher may run. More rounds = deeper research, higher cost. |
| `TOKEN_BUDGET_PER_RESEARCHER` | Token cap applied individually to each researcher agent via `TokenBudgetMiddleware`. |
| `MAX_COST_PER_RESEARCHER` | Dollar cap per researcher. Prevents one runaway thread from consuming the whole budget. |

In [ ]:
%pip install -q vidbyte-sdk python-dotenv requests pydantic

import json
import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

load_dotenv()
load_dotenv("../../.env")

PROVIDER  = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL     = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")
MAX_TOOL_ROUNDS              = 8
TOKEN_BUDGET_PER_RESEARCHER  = 20_000
MAX_COST_PER_RESEARCHER      = 0.30

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), \
    "Set OPENAI_API_KEY (or ANTHROPIC_API_KEY + overrides) in .env first."

## 2. System prompts

Three roles, three prompts. Each is focused on exactly one job:

- **Planner** — decompose the question into researchable sub-questions.
  Adapt the number of sub-questions to control research depth vs. cost.
- **Researcher** — gather source-attributed findings for one sub-question.
  The rule "every finding must carry the source title you actually read"
  is what prevents hallucinated citations.
- **Synthesizer** — write the report from findings only. The key constraint:
  no facts that weren't in the findings. Change the output format here (e.g.,
  "write as a Wikipedia-style article" or "write as a three-slide deck").

In [ ]:
PLANNER_PROMPT = """You are the planning stage of a deep-research pipeline.
Decompose the user's question into 3-5 sub-questions that are independently
researchable and jointly sufficient to answer it. Prefer questions about
mechanisms, evidence, and counterarguments over definitions."""

RESEARCHER_PROMPT = """You are a researcher assigned one sub-question.
Search the corpus, read the most relevant articles in full, and distill findings.
Every finding MUST come from an article you actually read in this session and
MUST carry that article's exact title as its source. Record gaps honestly."""

SYNTHESIZER_PROMPT = """You are the synthesis stage of a deep-research pipeline.
Write a well-structured markdown report answering the objective using ONLY the
provided findings. Cite the source title in parentheses after every factual claim.
Where findings conflict or gaps were reported, say so explicitly.
End with a 'Sources' section listing every title cited. Do not add facts not
in the findings."""

## 3. Tools

The researcher agents get two tools — `search` and `read_article` — over
Wikipedia's public API. This keeps the notebook keyless (no Bing/Google API
needed), while the tool interface is identical in shape to a production web-
browsing setup: search to find candidates, read to ingest them.

The planner and synthesizer get no tools — they operate only on text.

In [ ]:
import re
from vidbyte import tool

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS  = {"User-Agent": "vidbyte-cookbook-deep-research/0.1"}


@tool
def search(query: str) -> list[dict]:
    """Search Wikipedia for articles matching the query. Returns up to 5 results
    with title and a short snippet. Use read_article to read one in full."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "list": "search", "srsearch": query,
        "srlimit": 5, "format": "json",
    })
    resp.raise_for_status()
    return [
        {"title": h["title"], "snippet": re.sub(r"<[^>]+>", "", h["snippet"])}
        for h in resp.json()["query"]["search"]
    ]


@tool
def read_article(title: str) -> str:
    """Read the plain-text extract of one Wikipedia article by its exact title
    (truncated to 6 000 chars). Use the exact title returned by search()."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "prop": "extracts", "explaintext": 1,
        "titles": title, "format": "json", "redirects": 1,
    })
    resp.raise_for_status()
    pages = resp.json()["query"]["pages"]
    extract = next(iter(pages.values())).get("extract", "")
    return extract[:6_000] if extract else f"No article found for '{title}'."

## 4. Middleware

Each researcher agent runs its own independent tool loop and can call the
model many times. Without middleware, a single difficult sub-question could
consume the entire run's budget. We apply budget guards per researcher:

| Middleware | Role in deep research |
|---|---|
| `TokenBudgetMiddleware` | Hard token cap per researcher — prevents one thread from consuming the whole budget. |
| `CostBudgetMiddleware` | Dollar cap per researcher — predictable per-run cost regardless of question complexity. |
| `RuntimeLimitMiddleware` | Wall-clock + iteration limits — a stuck researcher can't block the rest of the pipeline. |

In [ ]:
from vidbyte.middleware import (
    CostBudgetMiddleware,
    RuntimeLimitMiddleware,
    TokenBudgetMiddleware,
)

researcher_middleware = [
    TokenBudgetMiddleware(max_tokens=TOKEN_BUDGET_PER_RESEARCHER),
    CostBudgetMiddleware(max_cost_usd=MAX_COST_PER_RESEARCHER),
    RuntimeLimitMiddleware(max_iterations=15, max_elapsed_seconds=90.0),
]

## 5. Constructing the agents

**Planner** — receives only the question, returns a `ResearchPlan`. No tools
needed; `output_schema` gives us the typed decomposition.

**Researcher** — one instance per sub-question (we fork from a base to avoid
re-declaring configuration). Has tools, has middleware, emits `ResearchNotes`
with per-finding source attribution via `output_schema`.

**Synthesizer** — sees only the typed findings, writes the final report. No
tools, no middleware needed — it's a pure writing task.

In [ ]:
from vidbyte import BaseAgent


class ResearchPlan(BaseModel):
    objective: str = Field(description="One sentence stating what the report must establish.")
    sub_questions: list[str] = Field(description="3-5 independent sub-questions that jointly cover the objective.")


class Finding(BaseModel):
    claim: str = Field(description="One specific factual finding relevant to the sub-question.")
    source_title: str = Field(description="Exact title of the article this finding came from.")


class ResearchNotes(BaseModel):
    sub_question: str
    findings: list[Finding] = Field(description="3-6 source-attributed findings.")
    gaps: str = Field(default="", description="What the corpus could not establish, if anything.")


def get_structured(reply, schema):
    payload = reply.metadata.get("structured")
    if payload is None:
        raise RuntimeError(f"{schema.__name__} validation failed: {reply.content[:200]}")
    return payload if isinstance(payload, schema) else schema.model_validate(payload)


# Planner — no tools, structured output only
planner = BaseAgent(
    name="planner",
    system_prompt=PLANNER_PROMPT,
    provider=PROVIDER, model_name=MODEL,
    output_schema=ResearchPlan,
)

# Researcher base — forked per sub-question to get a fresh history each time
researcher_base = BaseAgent(
    name="researcher",
    system_prompt=RESEARCHER_PROMPT,
    provider=PROVIDER, model_name=MODEL,
    tools=[search, read_article],
    middleware=researcher_middleware,
    max_tool_rounds=MAX_TOOL_ROUNDS,
    output_schema=ResearchNotes,
)

# Synthesizer — sees only typed findings, writes the report
synthesizer = BaseAgent(
    name="synthesizer",
    system_prompt=SYNTHESIZER_PROMPT,
    provider=PROVIDER, model_name=MODEL,
)

## 6. Running the pipeline

The three stages run in sequence. The planner's typed output feeds the
researcher loop; the researchers' typed findings feed the synthesizer.
Everything is typed at the boundaries — no prose-parsing between stages.

**Example questions you can try:**

```
# Business / history
"Did the Concorde fail for economic, technical, or political reasons?"

# Science
"How does mRNA vaccine technology differ from traditional vaccines, and
 what are its known limitations?"

# Technology
"What caused the 2021 Facebook outage, and how do BGP routing failures
 typically propagate?"

# Economics
"Why did Silicon Valley Bank fail in March 2023?"
```

In [ ]:
QUESTION = "Did the Concorde fail for economic, technical, or political reasons?"

# Stage 1 — plan
print("Planning...")
plan = get_structured(planner.run(QUESTION), ResearchPlan)
print(f"Objective: {plan.objective}")
for i, q in enumerate(plan.sub_questions, 1):
    print(f"  {i}. {q}")

# Stage 2 — research each sub-question (sequential; use ActorRuntime for parallel)
notes: list[ResearchNotes] = []
for q in plan.sub_questions:
    print(f"\nResearching: {q}")
    researcher = researcher_base.fork(name=f"researcher-{len(notes)+1}")
    result = get_structured(researcher.run(f"Research this sub-question: {q}"), ResearchNotes)
    notes.append(result)
    print(f"  → {len(result.findings)} findings")

# Stage 3 — synthesize
print("\nSynthesizing report...")
payload = {"objective": plan.objective, "research_notes": [n.model_dump() for n in notes]}
report = synthesizer.run("Write the report.\n\n" + json.dumps(payload, indent=2))
display(Markdown(report.content))

## 7. Example output

The synthesizer produces a structured Markdown report where every factual
claim has a source citation in parentheses. Below is a representative excerpt
for the Concorde question:

---

**Why the Concorde Failed — Economic, Technical, and Political Dimensions**

The Concorde's commercial failure resulted from the convergence of all three
factors, not any single cause.

**Economic:** The aircraft was only economically viable on routes where
ticket premiums offset its high operating costs (Concorde, Wikipedia). Fuel
costs, which consumed roughly four times more per seat-mile than subsonic
jets, made profitability outside a small luxury-traveller market impossible
(Concorde, Wikipedia).

**Technical:** The sonic boom restricted supersonic flight to oceanic routes,
eliminating most high-density city-pair routes (Sonic boom, Wikipedia).
The 2000 Air France crash, caused by titanium debris on the runway igniting
fuel, grounded the fleet for over a year and damaged public confidence
irreparably (Air France Flight 4590, Wikipedia).

**Political:** The programme survived for decades primarily due to a
binding treaty between the UK and French governments preventing unilateral
withdrawal, regardless of commercial outlook (Concorde, Wikipedia).

**Sources**
- Concorde — Wikipedia
- Sonic boom — Wikipedia
- Air France Flight 4590 — Wikipedia

---

*Adjust `MAX_TOOL_ROUNDS` and `TOKEN_BUDGET_PER_RESEARCHER` to trade depth for cost.*